In [ ]:
import scipy.io as sio
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

DATA_DIR = Path.cwd().parent.parent / "data" / "hyperspectral_oil_spill"
file_path = DATA_DIR / "GM01.mat"

try:
    mat_data = sio.loadmat(file_path)
except Exception as e:
    print(f"Error loading file. If it's a v7.3 mat file, you may need to use 'h5py' instead of 'scipy.io'. Error: {e}")
    exit()

# Print the keys to see how the author named the variables
print("Keys inside GM01.mat:")
for key in mat_data.keys():
    if not key.startswith('__'):  # Ignore Python/MATLAB metadata
        data_type = type(mat_data[key])
        shape = mat_data[key].shape if isinstance(mat_data[key], np.ndarray) else "N/A"
        print(f" - '{key}': Type {data_type}, Shape {shape}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

img = mat_data["img"]   # (1243, 684, 224)
gt_map = mat_data["map"] # (1243, 684)

unique_labels, counts = np.unique(gt_map, return_counts=True)
total_pixels = gt_map.size

print("--- Class Distribution ---")
for label, count in zip(unique_labels, counts):
    pct = (count / total_pixels) * 100
    print(f"Class {label}: {count:,} pixels ({pct:.2f}%)")

# Plot Mean Spectral Signature for each class
plt.figure(figsize=(10, 5))
for label in unique_labels:
    mask = (gt_map == label)
    # Average across all spatial pixels belonging to this class
    mean_spectrum = img[mask].mean(axis=0)
    plt.plot(mean_spectrum, label=f"Class {label} (Pixels: {count:,})", linewidth=1.8)

plt.title("Mean Spectral Profile per Class (GM01)")
plt.xlabel("Spectral Band Index (0 to 223)")
plt.ylabel("Reflectance / Radiance")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

img = mat_data["img"]   # (1243, 684, 224)
gt_map = mat_data["map"] # (1243, 684)

# Choose Red, Green, Blue bands (approx ~650nm, ~550nm, ~480nm in AVIRIS)
rgb_bands = [29, 19, 9]
rgb_raw = img[:, :, rgb_bands].astype(np.float32)

# Mask out negative/bad fill values
rgb_clean = np.where(rgb_raw < 0, 0, rgb_raw)

# Percentile contrast stretch per channel (2nd to 98th percentile)
rgb_enhanced = np.zeros_like(rgb_clean)
for c in range(3):
    channel = rgb_clean[:, :, c]
    p_low, p_high = np.percentile(channel[channel > 0], (2, 98))
    channel_clipped = np.clip(channel, p_low, p_high)
    rgb_enhanced[:, :, c] = (channel_clipped - p_low) / (p_high - p_low + 1e-8)

# Plot side-by-side
fig, axs = plt.subplots(1, 2, figsize=(14, 7))
axs[0].imshow(rgb_enhanced)
axs[0].set_title("Enhanced RGB Visual (Bands 29, 19, 9)")
axs[0].axis("off")

im = axs[1].imshow(gt_map, cmap="inferno")
axs[1].set_title("Ground Truth Oil Spill Mask")
axs[1].axis("off")
fig.colorbar(im, ax=axs[1], fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
import os

class HyperspectralDataset(Dataset):
    def __init__(self, file_paths, is_train=True):
        self.pixels = []
        self.labels = []
        
        for file_path in file_paths:
            mat_data = sio.loadmat(file_path)
            
            img = mat_data["img"]
            gt_map = mat_data["map"]
            
            img_flat = img.reshape(-1, img.shape[-1]).astype(np.float32)
            gt_flat = gt_map.reshape(-1).astype(np.int64)
            
            if is_train:
                water_mask = (gt_flat == 0)
                img_flat = img_flat[water_mask]
                gt_flat = gt_flat[water_mask]
                
            self.pixels.append(img_flat)
            self.labels.append(gt_flat)
            
        self.pixels = np.vstack(self.pixels)
        self.labels = np.concatenate(self.labels)

    def __len__(self):
        return len(self.pixels)

    def __getitem__(self, idx):
        pixel = self.pixels[idx]
        label = self.labels[idx]
        
        x_tensor = torch.tensor(pixel, dtype=torch.float32).unsqueeze(0)
        y_tensor = torch.tensor(label, dtype=torch.long)
        
        return x_tensor, y_tensor


DATA_DIR = Path.cwd().parent.parent / "data" / "hyperspectral_oil_spill"
all_files = sorted(list(DATA_DIR.glob("*.mat")))

train_files = all_files[:14]
test_files = all_files[14:]

train_dataset = HyperspectralDataset(train_files, is_train=True)
test_dataset = HyperspectralDataset(test_files, is_train=False)

BATCH_SIZE = 1024
NUM_WORKERS = os.cpu_count()

train_dataloader = DataLoader(dataset=train_dataset,
                              batch_size=BATCH_SIZE,
                              num_workers=NUM_WORKERS,
                              shuffle=True)

test_dataloader = DataLoader(dataset=test_dataset,
                             batch_size=BATCH_SIZE,
                             num_workers=NUM_WORKERS,
                             shuffle=False)

In [ ]:
from torchinfo import summary
from torch import nn
import copy

device = "cuda" if torch.cuda.is_available() else "cpu"

class Dummy1DAutoencoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv1d(in_channels=1, out_channels=16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2) 
        )
        self.decoder = nn.Sequential(
            nn.Upsample(scale_factor=2), 
            nn.Conv1d(in_channels=16, out_channels=1, kernel_size=3, padding=1)
        )

    def forward(self, x):
        return self.decoder(self.encoder(x))

model = Dummy1DAutoencoder().to(device)
summary(model, input_size=[32, 1, 224])

In [ ]:
def train_step(model: nn.Module, dataloader: torch.utils.data.DataLoader, loss_fn: nn.Module, optimizer: torch.optim.Optimizer, device):
    model.train()
    train_loss = 0 

    for batch, (X, _) in enumerate(dataloader):
        X = X.to(device)

        reconstructed = model(X)
        loss = loss_fn(reconstructed, X)
        train_loss += loss.item()

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    return train_loss / len(dataloader)


def test_step(model: nn.Module, dataloader: torch.utils.data.DataLoader, loss_fn: nn.Module, device):
    model.eval()
    test_loss = 0

    with torch.inference_mode():
        for batch, (X, _) in enumerate(dataloader):
            X = X.to(device)
            reconstructed = model(X)
            
            loss = loss_fn(reconstructed, X)
            test_loss += loss.item()

    return test_loss / len(dataloader)

In [ ]:
from tqdm.auto import tqdm

def train(model: nn.Module, train_dataloader: torch.utils.data.DataLoader, test_dataloader: torch.utils.data.DataLoader, optimizer: torch.optim.Optimizer, loss_fn: nn.Module, epochs: int, device):
          
    results = {"train_loss": [], "test_loss": []}
    
    best_test_loss = float('inf')
    best_model_wts = copy.deepcopy(model.state_dict())

    for epoch in tqdm(range(epochs)):
        train_loss = train_step(model=model,
                                dataloader=train_dataloader,
                                loss_fn=loss_fn,
                                optimizer=optimizer,
                                device=device)
                                
        test_loss = test_step(model=model,
                              dataloader=test_dataloader,
                              loss_fn=loss_fn,
                              device=device)

        print(f"Epoch: {epoch} | Train Loss: {train_loss:.4f} | Test Loss: {test_loss:.4f}") 

        if test_loss < best_test_loss:
            best_test_loss = test_loss
            best_model_wts = copy.deepcopy(model.state_dict())

        results["train_loss"].append(train_loss)
        results["test_loss"].append(test_loss)

    model.load_state_dict(best_model_wts)
    return model, results

In [ ]:
NUM_EPOCHS = 20
LEARNING_RATE = 0.001

loss_fn = nn.MSELoss()

optimizer = torch.optim.AdamW(params=model.parameters(), lr=LEARNING_RATE)

print(f"Starting training on {device} for {NUM_EPOCHS} epochs...")

best_model, results = train(model=model,
                            train_dataloader=train_dataloader,
                            test_dataloader=test_dataloader,
                            optimizer=optimizer,
                            loss_fn=loss_fn,
                            epochs=NUM_EPOCHS,
                            device=device)

print("Training complete! The best model weights have been restored.")